# Protein glycosylation: NAG on 5XAU

Attach an N-acetylglucosamine (NAG) to **ASN A3287** of the six-chain structure 5XAU and
build the glycoprotein topology.

Crystal structures are rarely complete: residues A2676-2701 have disordered side chains that
`pdb2gmx` cannot parametrise. Here they are truncated to a glycine backbone - a simple,
explicit choice for a demonstration; model the missing atoms instead for production work.

Runtime: about 2 minutes.

## Setup
Checks that PolyLego, GROMACS and AmberTools are available and pins the thread counts.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join("..", "common")))
from polylego_examples import *

# Pin threads for every step launched from this notebook (see examples/README.md).
os.environ.setdefault("OMP_NUM_THREADS", "8")
for k in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[k] = "1"
os.environ.setdefault("PYTHONHASHSEED", "0")

GMX = check_env()
os.environ["POLYLEGO_GMX"] = GMX
# GPU for the GROMACS runs: "auto" (use a compatible GPU if there is one, else CPU),
# "none" (CPU only), or a GPU id such as "0".
GPU = os.environ.get("POLYLEGO_GPU", "auto")
os.makedirs("output", exist_ok=True)

PolyLego   1.4.0  ($CONDA_PREFIX/lib/python3.10/site-packages/PolyLego)
Python     3.10.13
GROMACS    $CONDA_PREFIX/bin/gmx
AmberTools $CONDA_PREFIX/bin/antechamber
OMP_NUM_THREADS        8
OPENBLAS_NUM_THREADS   1
MKL_NUM_THREADS        1


## 1. Inputs

In [2]:
show_mol("input/NAG.sdf")
print(open("input/conjugation_spec.json").read())

[3D view - run the notebook in Jupyter to display it]


[
  {
    "modifier_file": "NAG.sdf",
    "mol_name": "NAG",
    "attach_protein_resnum": 3287,
    "attach_protein_atomname": "ND2",
    "attach_protein_occurrence": 1,
    "attach_modifier_idx": 0,
    "deletor_modifier_idx": 14,
    "deletor_protein_atomname": "HD21"
  }
]



## 2. Prepare the protein

In [3]:
strip_hetatm("input/5xau.pdb", "output/5xau_atom.pdb")
PROTEIN = backbone_only("output/5xau_atom.pdb", "output/5xau_protein.pdb",
                        chain="A", residues=range(2676, 2702))

## 3. Build the conjugate

In [4]:
run(["PolyLego-BuildConjugate", "--protein", PROTEIN,
     "--conjugations", "input/conjugation_spec.json", "--output_path", "output",
     "--gmx", GMX, "--gpu", GPU],
    log="output/conjugate.log")

$ PolyLego-BuildConjugate --protein output/5xau_protein.pdb --conjugations input/conjugation_spec.json --output_path output --gmx $CONDA_PREFIX/bin/gmx --gpu none


  PolyLego 1.4.0 | BuildConjugate
  1 conjugation site(s) on 5xau_protein.pdb
  protein topology (pdb2gmx)
  modifier parameters and cross-links
  vacuum energy minimisation
  closest heavy-atom distance after EM: 2.54 A
  Finished in 1 min 51 s
    output/conjugate_top/conjugate.top
    output/em/em/em.gro
  -> exit 0, 113 s


## 4. Result

In [5]:
TOP, GRO = "output/conjugate_top/conjugate.top", "output/em/em/em.gro"
show_gro(TOP, GRO, "output", ligand_resn=("NAG",))

[3D view - run the notebook in Jupyter to display it]


In [6]:
import numpy as np, parmed
def crosslinks(top, gro, modifier_resn):
    """Covalent bonds between the protein and each modifier, with their length after EM."""
    s = parmed.load_file(top, xyz=gro)
    rows = []
    for b in s.bonds:
        a1, a2 = b.atom1, b.atom2
        r1, r2 = a1.residue.name, a2.residue.name
        if (r1 in modifier_resn) != (r2 in modifier_resn):
            p, m = (a2, a1) if r1 in modifier_resn else (a1, a2)
            d = np.linalg.norm(np.array([p.xx, p.xy, p.xz]) - np.array([m.xx, m.xy, m.xz]))
            b0 = b.type.req if b.type is not None else float("nan")
            rows.append((f"{p.residue.name}{p.residue.number}:{p.name}",
                         f"{m.residue.name}:{m.name}", round(float(d), 3), round(float(b0), 3)))
    print(f"  {'protein atom':14s}    {'modifier':10s}  {'length':>7s}  {'b0':>6s}")
    for r in rows:
        print(f"  {r[0]:14s} -- {r[1]:10s}  {r[2]:6.3f} A  {r[3]:5.3f} A")
    return rows

_ = crosslinks(TOP, GRO, {"NAG"})

  protein atom      modifier     length      b0
  ASN593:ND2     -- NAG:C1       1.514 A  1.522 A
